# HarnessChandelier

*Harness the Chandelier — connect to the center of your conversation.*  

**Topic Drift Tracking for Long-Running Agent Conversations**

In real-world AI agent interactions, users naturally drift across multiple topics —
then return to what they originally wanted.

HarnessChandelier doesn't try to prevent drift.
It **tracks it** — and finds the topic the user kept coming back to.

Like a chandelier at the center of a hall, the dominant topic stays fixed
no matter how much the conversation moves around it.

Built on:
- **BERTopic** + cuML UMAP/HDBSCAN — GPU-accelerated topic extraction
- **cuGraph PageRank** — topic importance ranking
- **Temporal edge weighting** — time-aware topic transition graph

Built for **Harness Engineering** workflows where conversations drift across topics.

This example demonstrates a scenario where an AI assistant repeatedly fails to follow the user's original design specifications, causing the user to continuously restate their core requirements throughout the conversation.

In [ ]:
from harness_chandelier import HarnessChandelier
import time

live_messages = [
    "I want to build a Figma-like website with blue tones.",
    "The sidebar should be on the left side.",
    "Oh wait, I got an error. What does this mean?",
    "I said BLUE! Not green!",
    "Another error popped up again.",
    "Don't you remember? Figma-like website!",
    "It's blue! Just the top part!",
    "Error again. Same one as before.",
    "Like Figma. Auto design. That's it.",
    "JESUS. It's still not working.",
]

print("=== Real-time Intent Tracking ===\n")

In [ ]:
for i, message in enumerate(live_messages):
    # 실제로는 datetime.now() 그대로 쓰면 됨
    result = ranker.add_message(message)
    
    print(f"[{i+1:02d}] {message[:50]}...")
    
    if result:
        print(f"      → Main Topic: {result.main_topic}")
        print(f"      → PageRank: {result.pagerank.iloc[0]['pagerank']:.4f}")
    else:
        print(f"      → (accumulating...)")
    
    print()
    time.sleep(0.5)  # 실제 대화처럼 간격

In [3]:
ranker = HarnessChandelier(
    weights={"delta_time": 0.2}
)

result = ranker.fit(messages, timestamps=real_timestamps)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
print(f"Main Topic: {result.main_topic}")
print()
print("=== PageRank (Topic Importance) ===")
print(result.pagerank)

Main Topic: 0

=== PageRank (Topic Importance) ===
    vertex  pagerank
0        0  0.185863
1        3  0.134400
2        6  0.105720
3        1  0.091297
4        8  0.085628
5        2  0.084069
6        7  0.075642
7        4  0.075631
8        5  0.075123
9        9  0.043335
10      -1  0.043292


In [5]:
from collections import Counter

print("=== Topic Distribution ===")
counter = Counter(result.topic_labels)
for topic, count in sorted(counter.items()):
    print(f"Topic {topic}: {count} count")

print()
print("=== Topic per Message ===")
for i, (msg, topic) in enumerate(zip(messages, result.topic_labels)):
    print(f"[{i:02d}] Topic {topic}: {msg[:55]}...")

=== Topic Distribution ===
Topic -1: 4 count
Topic 0: 9 count
Topic 1: 6 count
Topic 2: 5 count
Topic 3: 5 count
Topic 4: 5 count
Topic 5: 4 count
Topic 6: 4 count
Topic 7: 3 count
Topic 8: 3 count
Topic 9: 3 count

=== Topic per Message ===
[00] Topic -1: Hey, I have this idea for a website. I want to build so...
[01] Topic 1: The sidebar needs to be on the left side, and at the bo...
[02] Topic 0: So to clarify, it's definitely blue tones only. Not gre...
[03] Topic 5: When a person inputs a description, the site should aut...
[04] Topic 2: I just tried running the code and it threw an error. Wh...
[05] Topic 1: The input box is in the middle of the page right now. I...
[06] Topic 2: I ran the code again and it's still showing an error. W...
[07] Topic 4: Don't you remember? I said the website should be like F...
[08] Topic 3: Ah another error popped up again. Do I really have to r...
[09] Topic 0: It's blue! I said blue website! Not the whole page, jus...
[10] Topic -1: About file u

In [6]:
# Final Summary
print("=== Dominant Topic Analysis ===")
main_topic_messages = [
    (i, msg) for i, (msg, topic) 
    in enumerate(zip(messages, result.topic_labels)) 
    if topic == result.main_topic
]

print(f"Main Topic: {result.main_topic} (PageRank: {result.pagerank.iloc[0]['pagerank']:.4f})")
print(f"Appears in {len(main_topic_messages)} out of {len(messages)} messages")
print()
print("Messages classified as Main Topic:")
for i, msg in main_topic_messages:
    print(f"  [{i:02d}] {msg[:70]}...")

=== Dominant Topic Analysis ===
Main Topic: 0 (PageRank: 0.1859)
Appears in 9 out of 51 messages

Messages classified as Main Topic:
  [02] So to clarify, it's definitely blue tones only. Not green, not purple....
  [09] It's blue! I said blue website! Not the whole page, just the top part ...
  [12] The layout should be centered, with the main content area in white but...
  [16] Why did you change the blue color? I specifically said blue tones, not...
  [20] Can we go back to basics? Blue tones on top, white content area, sideb...
  [23] Stop changing things I didn't ask you to change. Just fix what I asked...
  [27] The blue color on top should be a gradient from dark blue to medium bl...
  [35] Now the blue styling broke on mobile. It looks completely different on...
  [38] The top section blue color - it should match the shade #1E3A8A specifi...
